In [1]:
import os
import pandas as pd
import numpy as np
import librosa
from scipy.stats import skew, kurtosis
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Helper functions for audio preprocessing

def pre_emphasis(signal, alpha=0.97):
    """Apply pre-emphasis filter to the signal."""
    return np.append(signal[0], signal[1:] - alpha * signal[:-1])

def bandpass_filter(signal, sr=16000, low_cutoff=80, high_cutoff=7800):
    """Apply bandpass filter to the signal."""
    nyquist = sr // 2
    low = low_cutoff / nyquist
    high = high_cutoff / nyquist
    b, a = librosa.filters.butter(5, [low, high], btype='band')
    return librosa.filters.filtfilt(b, a, signal)

# Feature extraction function (provided by you)
def extract_features(audio_data, sr=16000, n_mfcc=40, n_fft=1024, hop_length=160, win_length=400, window='hann',
                     n_chroma=12, n_mels=13, n_bands=7, fmin=100, bins_per_octave=12, tempogram_win=384, spectral_roll_percent=0.95):
    try:
        # Apply Pre-emphasis
        y = pre_emphasis(audio_data, alpha=0.97)

        # Apply Bandpass Filter
        y = bandpass_filter(y, sr=sr)

        # Zero Crossing Rate & RMS
        zcr = librosa.feature.zero_crossing_rate(y, frame_length=win_length, hop_length=hop_length)
        rms = librosa.feature.rms(y=y, frame_length=win_length, hop_length=hop_length)

        # STFT and Spectral Features
        stft = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length))
        S, phase = librosa.magphase(stft)

        # MFCC and Deltas
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length,
                                   win_length=win_length, window=window)
        mfcc_delta = librosa.feature.delta(mfcc)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

        # Advanced Spectral Features
        spectral_centroid = librosa.feature.spectral_centroid(S=S, sr=sr)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        spectral_rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=spectral_roll_percent)
        spectral_contrast = librosa.feature.spectral_contrast(S=S, sr=sr, n_bands=n_bands, fmin=fmin)
        chroma = librosa.feature.chroma_stft(S=S, sr=sr, n_chroma=n_chroma)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
                                                 win_length=win_length, n_mels=n_mels)
        tonnetz = librosa.feature.tonnetz(y=y, sr=sr)

        # Temporal Features

        # Feature aggregation with statistics
        feature_functions = [
            (mfcc, 'mfcc', ['mean', 'std', 'skew', 'max', 'min']),
            (mfcc_delta, 'mfcc_delta', ['mean', 'std']),
            (mfcc_delta2, 'mfcc_delta2', ['mean', 'std']),
            (zcr, 'zcr', ['mean', 'std', 'max']),
            (rms, 'rms', ['mean', 'std', 'max', 'min']),
            (spectral_centroid, 'spectral_centroid', ['mean', 'std', 'max']),
            (spectral_bandwidth, 'spectral_bandwidth', ['mean', 'std', 'max']),
            (spectral_rolloff, 'spectral_rolloff', ['mean', 'std']),
            (spectral_contrast, 'spectral_contrast', ['mean', 'std', 'max']),
            (chroma, 'chroma', ['mean', 'std']),
            (mel_spec, 'mel_spec', ['mean', 'std']),
            (tonnetz, 'tonnetz', ['mean', 'std']),
        ]

        features = []
        feature_names = []

        for feat, feat_name, stats in feature_functions:
            for i in range(feat.shape[0]):
                if 'mean' in stats: 
                    features.append(np.mean(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_mean")
                if 'std' in stats: 
                    features.append(np.std(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_std")
                if 'skew' in stats: 
                    features.append(skew(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_skew")
                if 'kurt' in stats: 
                    features.append(kurtosis(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_kurt")
                if 'max' in stats: 
                    features.append(np.max(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_max")
                if 'min' in stats: 
                    features.append(np.min(feat[i]))
                    feature_names.append(f"{feat_name}_{i+1}_min")

        # Return both features and their names
        return np.array(features), feature_names
    except Exception as e:
        print(f"Error processing audio data: {e}")
        return None, None

def load_and_extract_features(audio_path, sr=16000):
    """Load audio file and extract features."""
    try:
        # Load audio file with librosa
        audio_data, sr = librosa.load(audio_path, sr=sr)
        
        # Extract features
        features, feature_names = extract_features(audio_data, sr=sr)
        
        return features, feature_names
    except Exception as e:
        print(f"Error processing file {audio_path}: {e}")
        return None, None

def process_dataset(df, audio_folder):
    """Process all audio files in the dataset."""
    features_list = []
    valid_indices = []
    feature_names = None
    
    print(f"Processing {len(df)} audio files...")
    
    for idx, row in df.iterrows():
        if idx % 50 == 0:
            print(f"Processing file {idx}/{len(df)}")
        
        audio_path = os.path.join(audio_folder, row['filename'])
        features, names = load_and_extract_features(audio_path)
        
        if features is not None:
            features_list.append(features)
            valid_indices.append(idx)
            if feature_names is None:
                feature_names = names
    
    # Create feature matrix
    X = np.vstack(features_list)
    
    # Get valid samples
    valid_df = df.iloc[valid_indices].reset_index(drop=True)
    
    return X, valid_df, feature_names

def train_xgboost_model(X_train, y_train, use_gpu=True):
    """Train XGBoost model with GPU acceleration if available."""
    # Create XGBoost regressor
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'n_estimators': 1000,
        'early_stopping_rounds': 50,
        'verbose': 100
    }
    
    # Add GPU parameters if requested
    if use_gpu:
        params['tree_method'] = 'gpu_hist'
        params['gpu_id'] = 0
    
    # Create model
    model = xgb.XGBRegressor(**params)
    
    # Split data for training and validation
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # Train model
    model.fit(
        X_train_split, y_train_split,
        eval_set=[(X_val, y_val)],
        verbose=True
    )
    
    # Evaluate on validation set
    val_preds = model.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    print(f"Validation RMSE: {val_rmse:.4f}")
    
    return model

def main():
    """Main function to run the model training and prediction pipeline."""
    # File paths
    train_csv = 'train.csv'
    test_csv = 'test.csv'
    audio_folder = 'audio_file'  # Update this to your actual audio folder path
    submission_csv = 'submission.csv'
    
    # Load training data
    train_df = pd.read_csv(train_csv)
    print(f"Loaded {len(train_df)} training samples")
    
    # Process training data
    X_train, valid_train_df, feature_names = process_dataset(train_df, audio_folder)
    y_train = valid_train_df['score'].values  # Assuming the label column is named 'score'
    
    print(f"Training with {X_train.shape[0]} valid samples and {X_train.shape[1]} features")
    
    # Train XGBoost model
    model = train_xgboost_model(X_train, y_train, use_gpu=True)
    
    # Load test data
    test_df = pd.read_csv(test_csv)
    print(f"Loaded {len(test_df)} test samples")
    
    # Process test data
    X_test, valid_test_df, _ = process_dataset(test_df, audio_folder)
    
    # Make predictions
    predictions = model.predict(X_test)
    
    # Ensure predictions are within the expected range [0, 5]
    predictions = np.clip(predictions, 0, 5)
    
    # Create submission dataframe
    submission_df = pd.DataFrame({
        'filename': valid_test_df['filename'],
        'score': predictions
    })
    
    # Save submission file
    submission_df.to_csv(submission_csv, index=False)
    print(f"Submission saved to {submission_csv}")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'